# Greeks: finite-difference accuracy and delta hedging

This notebook compares numerical estimates of the Black–Scholes Greeks with their analytical values. It varies the finite-difference bump to show the trade-off between truncation error (large bumps) and floating-point cancellation (very small bumps). It then builds a self-financing delta hedge across simulated stock paths and examines its terminal hedging error.

Run the cells from top to bottom. The simulation uses a fixed random seed so the results are repeatable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pricing_hedging.greeks import black_scholes_greeks
from pricing_hedging.greeks_numeric import interpolate_pde_delta, interpolate_pde_value, pde_greeks, pde_vega_rho, simulate_pde_delta_hedge
from pricing_hedging.pde.explicit_euler import explicit_euler
from pricing_hedging.pde.grid import Makegrid

S0 = 100.0
K = 100.0
r = 0.05
sigma = 0.20
tau = 1.0
option_type = "call"

print(f"Option: {option_type}; spot={S0:.2f}; strike={K:.2f}; rate={r:.1%}; volatility={sigma:.1%}; maturity={tau:.1f} year")

## Greeks extracted from a PDE price surface

The numerical values below come from an explicit-Euler Black–Scholes PDE price surface. Delta, Gamma, and calendar-time Theta are extracted directly with `pde_greeks`; Vega and Rho use PDE bump-and-reprice surfaces. The analytical Black–Scholes values are printed only as a reference for the PDE discretisation error.

The spot grid is chosen so the initial spot is an interior grid node, avoiding an additional interpolation step for this comparison.

In [ ]:
Smax = 400.0
M = 200
N = 2_000
S_grid, tau_grid = Makegrid(Smax, tau, M, N)
price_surface = explicit_euler(S_grid, tau_grid, K, r, sigma, option_type)
surface_greeks = pde_greeks(S_grid, tau_grid, price_surface)
bump_greeks = pde_vega_rho(S_grid, tau_grid, K, r, sigma, option_type)

spot_index = int(np.argmin(np.abs(S_grid - S0)))
if not np.isclose(S_grid[spot_index], S0):
    raise ValueError("Choose Smax and M so S0 is a PDE spot-grid node.")

numerical = {
    "delta": surface_greeks["delta"][-1, spot_index],
    "gamma": surface_greeks["gamma"][-1, spot_index],
    "theta": surface_greeks["theta"][-1, spot_index],
    "vega": bump_greeks["vega"][-1, spot_index],
    "rho": bump_greeks["rho"][-1, spot_index],
}
exact = black_scholes_greeks(sigma, r, tau, K, S0, option_type)

print("PDE-surface Greeks (analytical values shown only as references):")
for name in ("delta", "gamma", "vega", "theta", "rho"):
    print(f"{name:>5}: PDE={numerical[name]: .10f}, analytical={exact[name]: .10f}, absolute error={abs(numerical[name] - exact[name]):.3g}")

fig, axis = plt.subplots(figsize=(9, 4))
names = list(numerical)
axis.bar(names, [abs(numerical[name] - exact[name]) for name in names], color="tab:blue")
axis.set_title("Absolute error of PDE-surface Greeks")
axis.set_ylabel("Absolute error")
axis.grid(True, axis="y", alpha=0.3)
plt.show()

## Simulated delta-hedging portfolio

The hedge starts with an option value and delta interpolated from a numerical Black–Scholes PDE price surface, then rebalances the stock position on the selected time grid. At maturity, the terminal hedge error is the portfolio value minus the option payoff. The simulation assumes the realized volatility equals the volatility used to calculate the hedge and uses risk-neutral geometric Brownian motion.

In [ ]:
n_rebalances = 52
n_paths = 5_000
seed = 123
Smax = 400.0
M = 200
N = 2_000
solver = "explicit"

hedge = simulate_pde_delta_hedge(
    S0=S0,
    r=r,
    realized_sigma=sigma,
    hedge_sigma=sigma,
    tau=tau,
    K=K,
    option_type=option_type,
    n_rebalances=n_rebalances,
    n_paths=n_paths,
    Smax=Smax,
    M=M,
    N=N,
    solver=solver,
    seed=seed,
)

hedging_errors = hedge["hedging_error"]
rmse = float(np.sqrt(np.mean(hedging_errors**2)))
surface_greeks = pde_greeks(hedge['S'], hedge['tau'], hedge['price_surface'])
initial_pde_delta = interpolate_pde_delta(hedge['S'], hedge['tau'], surface_greeks['delta'], S0, tau)
print(f"PDE solver: {hedge['solver']}; grid: {M + 1} spot nodes x {N + 1} time nodes")
print(f"Initial option premium: {hedge['initial_option_price']:.4f}")
print(f"Initial PDE Delta: {initial_pde_delta:.6f}")
print(f"Paths: {n_paths:,}; rebalances per path: {n_rebalances}; seed: {seed}")
print(f"Terminal hedging error (portfolio value - payoff):")
print(f"  Mean:   {np.mean(hedging_errors): .4f}")
print(f"  Std:    {np.std(hedging_errors, ddof=1): .4f}")
print(f"  RMSE:   {rmse: .4f}")
print(f"  5% / median / 95%: {np.quantile(hedging_errors, [0.05, 0.50, 0.95])}")

sample_index = 0
sample_times = hedge["times"]
sample_spot = hedge["paths"][sample_index]
sample_portfolio = hedge["portfolio_values"][sample_index]
remaining_maturities = tau - sample_times
sample_option_values = np.array([
    interpolate_pde_value(hedge['S'], hedge['tau'], hedge['price_surface'], spot, maturity)
    for spot, maturity in zip(sample_spot, remaining_maturities)
])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(sample_times, sample_spot, color="tab:blue", label="Underlying spot")
axes[0].set_xlabel("Time (years)")
axes[0].set_ylabel("Spot price")
axes[0].set_title("One simulated underlying path")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(sample_times, sample_portfolio, label="Hedging portfolio", color="tab:green")
axes[1].plot(sample_times, sample_option_values, label="PDE option value", color="tab:orange", linestyle="--")
axes[1].set_xlabel("Time (years)")
axes[1].set_ylabel("Value")
axes[1].set_title(f"Portfolio and option value on sample path (error={hedging_errors[sample_index]:.3f})")
axes[1].grid(True, alpha=0.3)
axes[1].legend()
fig.tight_layout()
plt.show()

fig, axis = plt.subplots(figsize=(8, 5))
axis.hist(hedging_errors, bins=50, color="steelblue", edgecolor="white", alpha=0.85)
axis.axvline(0, color="black", linewidth=1.5, label="Zero error")
axis.axvline(np.mean(hedging_errors), color="tab:red", linestyle="--", label="Mean error")
axis.set_xlabel("Terminal hedging error (portfolio value - payoff)")
axis.set_ylabel("Number of paths")
axis.set_title("Terminal delta-hedging error across simulated paths")
axis.grid(True, axis="y", alpha=0.3)
axis.legend()
fig.tight_layout()
plt.show()